# 31 — Defenses and Privacy in Federated Learning — Lab

## Description

m30 showed that FedAvg has no Byzantine fault tolerance (a single client scaling its update by $\Lambda=50$ collapsed the global model) and that a curious server can reconstruct training data from a client's raw gradient. This lab implements the two defenses that fix the first problem and the two that fix the second. For Byzantine robustness, you implement **Multi-Krum**, the generalization of Krum that averages the $m$ most mutually consistent client updates instead of discarding all but one, and run it against the exact m30 scaling attack; you also implement Coordinate-wise Trimmed Mean as a cheaper alternative, and sweep Multi-Krum's $m$ parameter to chart its performance-robustness trade-off directly. For the server-side threat, you work through a numeric demonstration of **SecAgg**'s double-masking cancellation. For privacy of the released model itself, you apply **Local Differential Privacy** with the Gaussian mechanism to client updates and measure the accuracy cost of a given privacy budget $\epsilon$.

## Objectives

- State the Krum aggregation rule and the client-count condition $n \geq 2f+3$ under which its Byzantine fault tolerance guarantee holds.
- Implement Multi-Krum (with Krum recovered as its $m=1$ special case) and explain the performance-robustness trade-off controlled by $m$.
- Explain the Coordinate-wise Trimmed Mean algorithm as an alternative robust aggregation rule.
- Describe the SecAgg double-masking protocol and state exactly what information it hides from the server.
- Apply Local Differential Privacy to client update vectors using the Gaussian mechanism and measure the accuracy cost relative to the privacy budget $\epsilon$.
- Reason about the combined effect of layering robust aggregation, SecAgg, and LDP.

## Prerequisites

You must have completed the **m29** (Federated Learning Fundamentals) and **m30** (Attacks on Federated Learning) lectures and labs. You should understand the FedAvg weighted-average update rule, the Byzantine fault model, and the gradient-scaling attack from m30 (a malicious client submits $\Lambda \cdot g_{\mathrm{mal}}$ instead of its honest update $g_{\mathrm{mal}}$). You should also have read the **m31 lecture** covering Krum, Multi-Krum, Coordinate-wise Trimmed Mean, SecAgg, and the Gaussian mechanism for Local Differential Privacy.

## Required Software/Packages

- NumPy >= 1.20 for the aggregation rules and the toy federated optimization task.
- Matplotlib >= 3.3 for plotting.

Unlike m29 and m30, this lab does **not** require Flower, TensorFlow, or PyTorch — see Background/Context below for why.

## Environment/Setup Instructions

Run the following cell to import NumPy and Matplotlib and fix the random seed for reproducibility.

**What this cell does:** Imports NumPy and Matplotlib and fixes the random seed. No deep learning framework is needed for this lab (see Background/Context for why).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print("NumPy version:", np.__version__)
print("Setup complete.")

## Background/Context

In m29 you ran a real Flower FedAvg simulation on MNIST. In m30, measuring the gradient-scaling attack across several scale factors with full retraining would have been slow to repeat, so the lab switched to a lightweight analytical model of FedAvg's round-by-round dynamics instead, driven by the same attack formula ($\Lambda \cdot g_{\mathrm{mal}}$).

This lab goes one step further in the same direction, for the same reason: comparing four aggregation rules (FedAvg, Krum, Multi-Krum, Trimmed Mean), sweeping Multi-Krum's $m$ parameter over six values, and sweeping the privacy budget $\epsilon$ over five values would mean retraining a real CNN dozens of times if done the m29 way. Instead, this lab uses a small, honest **toy federated optimization task**: $n$ clients collaboratively minimize the quadratic loss $\mathcal{L}(\theta) = \frac{1}{2}\lVert \theta - \theta^{*}\rVert^2$ toward a fixed, known optimum $\theta^{*}$, by sending their local gradient estimates $\nabla\mathcal{L}(\theta) = \theta - \theta^{*}$, plus ordinary per-client sampling noise, to the server each round. The server updates $\theta \leftarrow \theta - \mathrm{lr} \cdot \mathrm{aggregate}(g_1, \ldots, g_n)$.

This keeps every aggregation rule's code exactly what you would drop into a real system unchanged — `krum_scores`, `multi_krum`, and `trimmed_mean` operate on raw gradient vectors regardless of where those vectors came from — while making every experiment fast enough to repeat many times without a GPU. Convergence is measured the same way a real loss curve would be: $\lVert \theta_t - \theta^{*}\rVert$, the distance remaining to the optimum, round by round. A malicious client still submits $\Lambda \cdot g_{\mathrm{mal}}$, the identical m30 attack, now against a server that may or may not be using a robust aggregation rule.

## Exercise Instructions

**Part A — Byzantine-Robust Aggregation (20 min)**

1. Set up the toy federated optimization task: a fixed optimum $\theta^{*}$, $n=10$ clients, and functions that generate an honest client's gradient (true gradient plus noise) and a malicious client's gradient ($\Lambda$ times an honest one, the m30 attack). (4 min)
2. Implement `krum_scores`, `multi_krum` (with `krum` as the $m=1$ special case), `trimmed_mean`, and `fedavg`. (6 min)
3. Run FedAvg, Krum, Multi-Krum ($m=n-f$), and Trimmed Mean ($\beta=f$) for 10 rounds with $f=2$ malicious clients scaling by $\Lambda=50$. Plot $\lVert \theta_t - \theta^{*}\rVert$ for all four and report the final distance for each. (6 min)
4. Sweep Multi-Krum's $m$ from 1 to 10: once with all clients honest (to see the statistical-efficiency side of the trade-off) and once under the same attack as step 3 (to see the robustness side break once $m$ exceeds $n-f$). (4 min)

**Part B — Secure Aggregation (10 min)**

5. Implement the double-masking formula on 5 toy clients: pairwise masks that cancel exactly when summed, and a self-mask that does not. Confirm the server recovers $\sum_u x_u$ only after the self-masks are removed. (5 min)
6. Implement a small Shamir secret-sharing example and confirm that $t$ shares reconstruct a secret exactly while $t-1$ shares are consistent with *any* candidate secret. (5 min)

**Part C — Local Differential Privacy (10 min)**

7. Implement `clip_and_noise`: clip a gradient to L2 norm $C$, then add Gaussian noise calibrated by $\sigma = C\sqrt{2\ln(1.25/\delta)}/\epsilon$. Print $\sigma$ for $\epsilon \in \{0.5, 1, 2, 5, 10\}$ at fixed $C, \delta$ and confirm $\sigma \propto 1/\epsilon$. (4 min)
8. Re-run the toy task with all clients honest, applying LDP to every client's gradient, for each $\epsilon$ in that set plus a no-DP baseline. Plot the final distance to $\theta^{*}$ against $\epsilon$ — the accuracy cost of the privacy budget. (6 min)

## Part A: Byzantine-Robust Aggregation — Krum, Multi-Krum, Trimmed Mean

**What this cell does:** Defines the toy federated task. `D` is the toy parameter dimension (standing in for a flattened model's weights, as in m29/m30). `THETA_STAR` is the fixed, known optimum the clients are collectively trying to reach; `theta_init` is where the global model starts. `honest_gradient` returns the true gradient of $\mathcal{L}(\theta)=\frac12\lVert\theta-\theta^{*}\rVert^2$, which is $\theta-\theta^{*}$, plus Gaussian sampling noise (every honest client's local data gives a slightly different noisy estimate, exactly as in real FL). `malicious_gradient` is the identical m30 attack: $\Lambda$ times an otherwise-honest gradient.

In [ ]:
D = 30                      # toy parameter dimension (stands in for flattened model weights)
N = 10                      # total clients, matching m29/m30
F = 2                       # assumed number of Byzantine clients; n=10 >= 2f+3=7, so Krum's guarantee holds
SCALE = 50                  # the m30 attack's scale factor Lambda
LR = 0.2                    # server-side step size
NOISE_SCALE = 0.4           # per-client gradient sampling noise

THETA_STAR = np.zeros(D)
theta_init = np.random.randn(D)

def honest_gradient_at(theta):
    return (theta - THETA_STAR) + NOISE_SCALE * np.random.randn(D)

def malicious_gradient_at(theta, scale_factor=SCALE):
    '''The m30 attack: send Lambda times an otherwise-honest gradient.'''
    return scale_factor * honest_gradient_at(theta)

print(f"D={D}, N={N}, F={F}, SCALE={SCALE}")
print(f"Byzantine-tolerance check: n >= 2f+3  ->  {N} >= {2*F+3}  ->  {N >= 2*F+3}")

**The formulas.** Given $n$ client gradients $v_1, \ldots, v_n$ and an assumed bound $f$ on the number of Byzantine clients:

$$\mathrm{score}(i) = \sum_{j \in \mathcal{N}_i} \lVert v_i - v_j \rVert^2, \qquad \mathcal{N}_i = \text{the } (n-f-2) \text{ closest of the other } n-1 \text{ vectors to } v_i.$$

**Krum** keeps only the single lowest-scoring vector: $\mathrm{Krum}(v_1,\ldots,v_n) = v_{i^{*}}$, $i^{*}=\arg\min_i \mathrm{score}(i)$.

**Multi-Krum** generalizes this: sort all $n$ vectors by score and average the $m$ with the lowest scores:

$$\mathrm{MultiKrum}_m(v_1, \ldots, v_n) = \frac{1}{m}\sum_{i \in S} v_i, \qquad S = \text{the } m \text{ indices with the smallest score}(i).$$

$m=1$ recovers Krum exactly; $m=n$ discards no information at all and recovers the plain FedAvg mean. The default, Byzantine-safe choice is $m=n-f$.

**Coordinate-wise Trimmed Mean** instead filters each coordinate $j$ independently: sort the $n$ clients' values on coordinate $j$, drop the $\beta$ largest and $\beta$ smallest, and average what remains:

$$\mathrm{TrimmedMean}_j(v_1,\ldots,v_n) = \frac{1}{n-2\beta}\sum_{i \,\in\, \mathrm{middle}(n-2\beta)} v_{i,j}.$$

**What this cell does:** Implements the four aggregation rules exactly as written above. `pairwise_sq_distances` is a vectorized $n \times n$ matrix of squared Euclidean distances, used by `krum_scores`. `multi_krum(updates, f, m)` returns both the aggregated vector and the indices it selected, so later cells can report *which* clients were kept each round. `krum` is defined as `multi_krum` with `m=1` — not a separate implementation, exactly because Krum *is* that special case. `trimmed_mean` sorts each coordinate independently with `np.sort(..., axis=0)` and averages the middle slice. Two assertions confirm the $m=1$ and $m=n$ special cases behave exactly as the formulas say they must.

In [ ]:
def pairwise_sq_distances(V):
    '''n x n matrix of squared Euclidean distances between the rows of V.'''
    diffs = V[:, None, :] - V[None, :, :]
    return np.sum(diffs ** 2, axis=-1)

def krum_scores(V, f):
    '''Krum score for every vector: sum of squared distances to its (n-f-2) nearest neighbors.'''
    n = len(V)
    k = n - f - 2
    assert k >= 1, "Need n >= f + 3 for Krum's score to be defined."
    D_mat = pairwise_sq_distances(V)
    scores = np.zeros(n)
    for i in range(n):
        dists_to_others = np.delete(D_mat[i], i)
        scores[i] = np.sort(dists_to_others)[:k].sum()
    return scores

def multi_krum(V, f, m=None):
    '''Average the m lowest-scoring vectors. Defaults to m = n - f (the safe choice).'''
    n = len(V)
    if m is None:
        m = n - f
    scores = krum_scores(V, f)
    selected = np.argsort(scores)[:m]
    return V[selected].mean(axis=0), selected

def krum(V, f):
    '''Krum is Multi-Krum with m=1 -- not a separate algorithm.'''
    agg, selected = multi_krum(V, f, m=1)
    return agg, selected

def trimmed_mean(V, beta):
    '''Per-coordinate: drop the beta largest and beta smallest values, average the rest.'''
    n = len(V)
    assert 2 * beta < n, "Need 2*beta < n."
    S = np.sort(V, axis=0)
    return S[beta:n - beta].mean(axis=0)

def fedavg(V):
    return V.mean(axis=0)

# Sanity checks: the two named special cases of Multi-Krum.
_V_test = np.random.randn(N, D)
_agg_m1, _ = multi_krum(_V_test, F, m=1)
_agg_krum, _ = krum(_V_test, F)
assert np.allclose(_agg_m1, _agg_krum), "multi_krum(V, f, m=1) must equal krum(V, f)."
_agg_mn, _ = multi_krum(_V_test, F, m=N)
assert np.allclose(_agg_mn, fedavg(_V_test)), "multi_krum(V, f, m=n) must equal the plain mean."
print("PASS: multi_krum(m=1) == krum, and multi_krum(m=n) == fedavg mean.")

**What this cell does:** Runs the toy federated task for 10 rounds under the m30 attack ($f=2$ clients out of 10 scaling by $\Lambda=50$), once per aggregation rule. `simulate_fl` is the server loop: each round it draws $n-f$ honest gradients and $f$ malicious ones at the *current* $\theta$, applies the chosen `aggregate_fn`, and takes a gradient step. It returns the distance $\lVert\theta_t-\theta^{*}\rVert$ after every round. The same random seed is reused across the four runs so the only thing that differs between curves is the aggregation rule, not the random draws.

In [ ]:
def simulate_fl(aggregate_fn, num_rounds, n=N, f=F, scale_factor=SCALE, seed=0):
    '''Run the toy federated task for num_rounds, returning ||theta_t - theta*|| after each round.'''
    rng_state = np.random.get_state()
    np.random.seed(seed)
    theta = theta_init.copy()
    distances = [np.linalg.norm(theta - THETA_STAR)]
    for _ in range(num_rounds):
        malicious = [malicious_gradient_at(theta, scale_factor) for _ in range(f)]
        honest = [honest_gradient_at(theta) for _ in range(n - f)]
        updates = np.stack(malicious + honest)   # client indices 0..f-1 are malicious
        aggregate = aggregate_fn(updates)
        theta = theta - LR * aggregate
        distances.append(np.linalg.norm(theta - THETA_STAR))
    np.random.set_state(rng_state)
    return distances

NUM_ROUNDS = 10
results = {
    "FedAvg (no defense)": simulate_fl(fedavg, NUM_ROUNDS),
    "Krum (m=1)": simulate_fl(lambda V: krum(V, F)[0], NUM_ROUNDS),
    f"Multi-Krum (m={N-F})": simulate_fl(lambda V: multi_krum(V, F)[0], NUM_ROUNDS),
    f"Trimmed Mean (beta={F})": simulate_fl(lambda V: trimmed_mean(V, F), NUM_ROUNDS),
}

plt.figure(figsize=(9, 6))
for label, dist_curve in results.items():
    plt.plot(range(len(dist_curve)), dist_curve, marker='o', label=label)
plt.xlabel('Round')
plt.ylabel('Distance to optimum  ||theta_t - theta*||')
plt.title(f'Byzantine-robust aggregation under the m30 scaling attack (f={F}, Lambda={SCALE})')
plt.yscale('log')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("Final distance to theta* after", NUM_ROUNDS, "rounds:")
for label, dist_curve in results.items():
    print(f"  {label:28s}: {dist_curve[-1]:.4f}")

Expected output: the FedAvg curve should grow across rounds (the attack drags $\theta$ away from $\theta^{*}$ without bound), while Krum, Multi-Krum, and Trimmed Mean should all shrink toward 0, with Krum visibly noisier/slower than Multi-Krum and Trimmed Mean (it is using only 1 of the 10 clients' information every round, instead of $n-f=8$).

**What this cell does:** Sweeps Multi-Krum's $m$ parameter from 1 to 10 in two separate conditions, to chart both sides of the performance-robustness trade-off. `sweep_m_honest_only` runs every client honestly (no attacker) and averages the final distance over 30 random seeds, isolating how much averaging more vectors ($m$ larger) reduces variance. `sweep_m_under_attack` repeats the exact same attack as the previous cell ($f=2$, $\Lambda=50$) for every $m$, showing that the defense holds while $m \leq n-f=8$ and then degrades once $m$ is forced to include a malicious vector.

In [ ]:
def sweep_m_honest_only(m_values, num_rounds=15, num_trials=30):
    means = []
    for m in m_values:
        finals = [simulate_fl(lambda V: multi_krum(V, F, m=m)[0], num_rounds, f=0, seed=s)[-1]
                  for s in range(num_trials)]
        means.append(np.mean(finals))
    return means

def sweep_m_under_attack(m_values, num_rounds=10, num_trials=10):
    means = []
    for m in m_values:
        finals = [simulate_fl(lambda V: multi_krum(V, F, m=m)[0], num_rounds, seed=s)[-1]
                  for s in range(num_trials)]
        means.append(np.mean(finals))
    return means

m_values = [1, 2, 4, 6, 8, 10]
honest_curve = sweep_m_honest_only(m_values)
attack_curve = sweep_m_under_attack(m_values)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(m_values, honest_curve, marker='o', color='tab:blue')
axes[0].set_xlabel('m (number of vectors averaged)')
axes[0].set_ylabel('Final distance to theta* (no attack)')
axes[0].set_title('Statistical efficiency: bigger m = less variance')
axes[0].grid(True, alpha=0.3)

axes[1].plot(m_values, attack_curve, marker='s', color='tab:red')
axes[1].axvline(N - F, color='gray', linestyle='--', label=f'm = n - f = {N - F}')
axes[1].set_xlabel('m (number of vectors averaged)')
axes[1].set_ylabel('Final distance to theta* (under m30 attack)')
axes[1].set_title('Robustness: breaks once m > n - f')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Honest-only final distance by m:", [round(v, 3) for v in honest_curve])
print("Under-attack final distance by m:", [round(v, 3) for v in attack_curve])

Expected output: the left panel should decrease roughly monotonically as $m$ grows (more averaging, less variance, even with no attacker). The right panel should stay small for $m \leq 8 = n-f$ and then jump sharply at $m=10$ (the next value this sweep tests after 8), the point at which Multi-Krum is forced to average in at least one of the two malicious vectors.

## Part B: Secure Aggregation — Double-Masking Demonstration

**Recap of the formula.** Each toy client $u$ sends $y_u = x_u + b_u + \sum_{v>u} s_{u,v} - \sum_{v<u} s_{u,v}$, where $b_u$ is $u$'s own random self-mask and $s_{u,v}=s_{v,u}$ is a secret shared only between $u$ and $v$. When every client submits, summing all $y_u$ makes every pairwise term cancel automatically (the real protocol uses `PRG(s_uv)` and modular arithmetic; this toy demo uses plain random vectors and real addition, which is enough to see the cancellation), leaving $\sum_u y_u = \sum_u x_u + \sum_u b_u$ — the self-masks are the only thing left for the server to remove.

**What this cell does:** Builds 5 toy clients, each with a secret contribution `x_u`, a random self-mask `b_u`, and symmetric pairwise masks `s[u, v] == s[v, u]`. It computes each client's masked message `y_u` using the sign convention above, sums all five, and checks the two claims from the recap: the sum equals `sum(x) + sum(b)` exactly (pairwise masks cancel on their own), and that sum is *not* equal to `sum(x)` alone (the self-masks genuinely remain, and must be removed by some other mechanism — which is exactly what Shamir secret sharing is for, demonstrated in the next cell).

In [ ]:
N_TOY = 5
DIM_TOY = 4

x = {u: np.random.randn(DIM_TOY) for u in range(N_TOY)}        # each client's true secret contribution
b = {u: np.random.randn(DIM_TOY) for u in range(N_TOY)}        # each client's self-mask

s = {}                                                          # symmetric pairwise masks: s[u,v] == s[v,u]
for u in range(N_TOY):
    for v in range(u + 1, N_TOY):
        shared = np.random.randn(DIM_TOY)
        s[(u, v)] = shared
        s[(v, u)] = shared

def masked_message(u):
    y = x[u] + b[u]
    for v in range(N_TOY):
        if v == u:
            continue
        y = y + s[(u, v)] if v > u else y - s[(u, v)]
    return y

y = {u: masked_message(u) for u in range(N_TOY)}
server_sum = sum(y.values())

sum_x = sum(x.values())
sum_b = sum(b.values())

print("Does server_sum == sum(x) + sum(b)?  ", np.allclose(server_sum, sum_x + sum_b))
print("Does server_sum == sum(x) alone?     ", np.allclose(server_sum, sum_x))
print("\n-> Pairwise masks cancel for free. Self-masks remain and must be removed separately")
print("   (via Shamir secret sharing, so no single party -- not even the server -- can remove")
print("   them without cooperation from a threshold number of the other clients).")

**What this cell does:** Implements a small Shamir $(t, n)$ secret-sharing scheme over real numbers (production SecAgg uses a finite field for information-theoretic security; this version is for intuition only, as noted in the lecture). `make_shares` builds a random degree-$(t-1)$ polynomial whose constant term is the secret, and evaluates it at $n$ points to produce $n$ shares. `reconstruct` uses Lagrange interpolation on any $t$ of those shares to recover the polynomial's constant term exactly. The cell then shows that $t-1$ shares are consistent with *any* candidate secret: with only $t-1$ equations and $t$ unknown coefficients, the system is underdetermined, so you can always solve for a coefficient that makes some other chosen "secret" fit the same $t-1$ points equally well.

In [ ]:
def make_shares(secret, t, n, seed=0):
    '''Random degree-(t-1) polynomial with constant term `secret`; returns n (x, y) shares.'''
    rng = np.random.RandomState(seed)
    coeffs = [secret] + list(rng.randn(t - 1))   # coeffs[0] is the secret
    xs = np.arange(1, n + 1)
    ys = np.array([sum(c * (x_val ** i) for i, c in enumerate(coeffs)) for x_val in xs])
    return list(zip(xs, ys))

def reconstruct(shares):
    '''Lagrange interpolation at x=0 to recover the polynomial's constant term (the secret).'''
    total = 0.0
    for i, (xi, yi) in enumerate(shares):
        term = yi
        for j, (xj, _) in enumerate(shares):
            if i != j:
                term *= (0 - xj) / (xi - xj)
        total += term
    return total

T, N_SHARES = 3, 5
SECRET = 42.0
shares = make_shares(SECRET, T, N_SHARES, seed=1)
print("All shares:", [(x_, round(y_, 3)) for x_, y_ in shares])

recovered = reconstruct(shares[:T])
print(f"\nReconstructing from {T} shares: {recovered:.6f}  (true secret: {SECRET})")
assert abs(recovered - SECRET) < 1e-6

# With only t-1 = 2 shares, show that a DIFFERENT candidate secret is equally consistent.
partial_shares = shares[:T - 1]
candidate_secret = SECRET + 1000.0   # an arbitrary, very different "secret"
(x1, y1), (x2, y2) = partial_shares
# Solve for a 2-coefficient line through (x1,y1) and (x2,y2) that passes through (0, candidate_secret):
# with 3 points (0, candidate_secret), (x1,y1), (x2,y2) and a degree-2 polynomial, there's always a
# unique quadratic through any 3 non-colinear points -- so some valid polynomial exists for ANY candidate.
fake_poly_points = [(0, candidate_secret), (x1, y1), (x2, y2)]
fake_recovered = reconstruct(fake_poly_points)
print(f"\nWith only {T-1} shares, candidate secret {candidate_secret} fits a valid")
print(f"degree-{T-1} polynomial through those points just as well: reconstruct(...) = {fake_recovered:.3f}")
print("-> t-1 shares reveal nothing: any candidate secret is consistent with some valid polynomial.")

Expected output: reconstructing from $t=3$ shares recovers `42.0` exactly. The final check confirms that with only $t-1=2$ shares, an arbitrary, very different candidate secret can be made to fit a valid polynomial through those same points just as well — which is exactly why SecAgg can give surviving clients' secret shares to *some* of the other clients (for dropout recovery) without ever letting fewer than $t$ of them learn anything about the secret itself.

## Part C: Local Differential Privacy — the Gaussian Mechanism

**Recap of the formula.** Each client clips its gradient to L2 norm $C$, then adds noise $\mathcal{N}(0,\sigma^2 I)$ with $\sigma = C\sqrt{2\ln(1.25/\delta)}/\epsilon$ (the classical Gaussian mechanism, valid for $\epsilon \in (0,1)$ and commonly used somewhat more broadly in practice). Smaller $\epsilon$ (stronger privacy) means larger $\sigma$ (more noise).

**What this cell does:** Implements `clip_and_noise` exactly as in the recap, and prints $\sigma$ for five values of $\epsilon$ at a fixed clip bound $C=1.5$ and $\delta=10^{-5}$, confirming $\sigma \propto 1/\epsilon$.

In [ ]:
CLIP_NORM = 1.5
DELTA = 1e-5

def clip_and_noise(g, clip_norm=CLIP_NORM, eps=None, delta=DELTA):
    '''Clip g to L2 norm clip_norm, then add Gaussian noise calibrated to (eps, delta)-LDP.'''
    norm = np.linalg.norm(g)
    clipped = g * min(1.0, clip_norm / (norm + 1e-12))
    if eps is None:          # eps=None means "no DP noise" -- the baseline for comparison
        return clipped
    sigma = clip_norm * np.sqrt(2 * np.log(1.25 / delta)) / eps
    return clipped + np.random.normal(0, sigma, size=g.shape)

print(f"clip_norm={CLIP_NORM}, delta={DELTA}\n")
print(f"{'epsilon':>8} | {'sigma':>8}")
for eps in [0.5, 1.0, 2.0, 5.0, 10.0]:
    sigma = CLIP_NORM * np.sqrt(2 * np.log(1.25 / DELTA)) / eps
    print(f"{eps:>8.1f} | {sigma:>8.3f}")
print("\nsigma is exactly proportional to 1/epsilon, as the formula says.")

**What this cell does:** Re-runs the toy federated task from Part A, this time with **no** malicious clients, but with LDP applied to every honest client's gradient before the server averages them (`simulate_fl_ldp` is `simulate_fl` with `clip_and_noise` inserted between generating each gradient and aggregating). It sweeps $\epsilon \in \{0.5, 1, 2, 5, 10\}$ plus a no-DP baseline, averaging the final distance over 30 seeds per $\epsilon$ for a smooth curve, and plots the accuracy cost (distance to $\theta^{*}$) against the privacy budget.

In [ ]:
def simulate_fl_ldp(num_rounds, eps, num_trials=30, clip_norm=CLIP_NORM, delta=DELTA):
    finals = []
    for seed in range(num_trials):
        rng_state = np.random.get_state()
        np.random.seed(seed)
        theta = theta_init.copy()
        for _ in range(num_rounds):
            grads = np.stack([
                clip_and_noise(honest_gradient_at(theta), clip_norm=clip_norm, eps=eps, delta=delta)
                for _ in range(N)
            ])
            theta = theta - LR * fedavg(grads)
        finals.append(np.linalg.norm(theta - THETA_STAR))
        np.random.set_state(rng_state)
    return np.mean(finals)

eps_values = [0.5, 1.0, 2.0, 5.0, 10.0]
ldp_finals = [simulate_fl_ldp(num_rounds=15, eps=eps) for eps in eps_values]
no_dp_final = simulate_fl_ldp(num_rounds=15, eps=None)

plt.figure(figsize=(8, 5))
plt.plot(eps_values, ldp_finals, marker='o', label='With LDP (Gaussian mechanism)')
plt.axhline(no_dp_final, color='gray', linestyle='--', label='No DP (baseline)')
plt.xlabel('Privacy budget epsilon (smaller = more private)')
plt.ylabel('Final distance to theta* (15 rounds, honest only)')
plt.title('Accuracy cost of Local Differential Privacy')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("Final distance by epsilon:")
for eps, final in zip(eps_values, ldp_finals):
    print(f"  epsilon={eps:>4.1f}: {final:.4f}")
print(f"  no DP     : {no_dp_final:.4f}")

Expected output: the final distance should decrease (closer to the no-DP baseline) as $\epsilon$ grows from 0.5 to 10, with $\epsilon=0.5$ noticeably worse than the no-DP run and $\epsilon=10$ close to it — the accuracy cost of privacy, directly as a function of the privacy budget.

## Comprehension Questions

1. Krum's Byzantine fault tolerance guarantee requires $n \geq 2f+3$. In your own words, what could go wrong if this lab used $n=10$ clients with $f=4$ malicious ones instead of $f=2$?
2. In the Part A plots, Krum ($m=1$) converges more slowly and noisily than Multi-Krum ($m=n-f=8$), even though both successfully resist the attack. Why does keeping only one client's update per round cost statistical efficiency, and why does Multi-Krum's larger $m$ not reopen the same vulnerability that FedAvg's $m=n$ has?
3. Coordinate-wise Trimmed Mean and Multi-Krum both discard outliers, but by different mechanisms. What is the key structural difference between how they decide what to discard, and why can Trimmed Mean's output be a vector that no single client actually sent?
4. In Part B, the server only ever sees `server_sum`, `sum(x) + sum(b)` for all five clients combined. Name one specific thing about any individual client's `x_u` that this protocol guarantees the server can never learn, even in principle, from that sum alone.
5. In Part C, why does averaging $n$ independently LDP-noised client gradients not make the noise disappear entirely, the way it would for ordinary per-client sampling noise with no adversarial or privacy motivation behind it?
6. Suppose a deployment wanted to run Multi-Krum *and* SecAgg together in their basic forms, as both are implemented in this lab. Explain concretely why the server could not run the `krum_scores` function from Part A on the masked values it receives under SecAgg from Part B.

**Answer 1:** With $f=4$ Byzantine clients among $n=10$, the condition $n \geq 2f+3 = 11$ fails. The four malicious clients could then coordinate so that each of them sits among the others' nearest neighbors, manufacturing artificially low Krum scores for themselves (since much of each malicious client's "$n-f-2$ closest neighbors" window could be filled by its fellow attackers instead of genuine honest clients). Krum's guarantee no longer applies once this is possible, and the lowest-scoring vector, the one Krum would select, could end up being one of the malicious ones instead of an honest one.

**Answer 2:** Krum discards 9 of the 10 clients' gradients every round and keeps only 1, so the result still carries that one client's full sampling noise with none of the variance reduction that averaging multiple independent noisy estimates would normally provide — the same cost a mean of 1 sample always pays relative to a mean of 8. Multi-Krum with $m=8=n-f$ averages 8 vectors instead of 1, reducing variance roughly as $1/\sqrt{8}$ relative to Krum, without reopening FedAvg's vulnerability, because the 8 selected vectors are chosen by the *same* distance-based scoring that (under $n\geq2f+3$) keeps the 2 malicious vectors out of the selected set in the first place. The difference between $m=8$ and FedAvg's $m=10$ is exactly those 2 excluded vectors.

**Answer 3:** Multi-Krum scores and discards whole *vectors*: it looks at each client's full update as one point and keeps or drops it entirely based on its distance to its nearest neighbors in the full-dimensional space. Trimmed Mean instead scores and discards *per coordinate*: on coordinate 1 it might discard clients 3 and 7 as the extremes, while on coordinate 2 it discards clients 1 and 9. Because the discarding decision is made independently per coordinate, the surviving average on coordinate 1 can come from a different subset of clients than the surviving average on coordinate 2 — so the final vector is a coordinate-by-coordinate blend that no single client, honest or otherwise, ever actually produced as one complete update.

**Answer 4:** The server can never learn any individual client's raw contribution $x_u$ by itself — not its value, not even which of the five clients contributed which part of the sum. `server_sum` (even after the self-masks are eventually removed through secret sharing) is one single aggregate number for all five clients together; there is no operation available to the server that isolates one client's term from that sum, because the masking was specifically constructed so that only the full combination, never a partial one, cancels to something meaningful.

**Answer 5:** Averaging $n$ independent LDP noise draws reduces the noise's *variance* by a factor of $n$ (so its standard deviation shrinks by $\sqrt{n}$), but the noise was deliberately scaled, via $\sigma = C\sqrt{2\ln(1.25/\delta)}/\epsilon$, to be large enough that even after this $1/\sqrt{n}$ reduction, it still obscures enough about any single client's contribution to satisfy the $(\epsilon,\delta)$ privacy guarantee for that client. Ordinary sampling noise has no such floor enforced on it; LDP noise's whole purpose is to leave a privacy-preserving residual behind no matter how many clients' noise gets averaged together.

**Answer 6:** `krum_scores` needs the actual numerical values of each client's individual gradient vector, to compute pairwise distances `pairwise_sq_distances(V)` between them. Under SecAgg (Part B), the server never receives any such vector; it only ever receives the sum of all clients' masked messages, `server_sum`, a single combined number for the whole group. There is no list of individual vectors `V` for `krum_scores` to index into — SecAgg's entire design goal was to make sure that list never reaches the server in the first place, so the function has no valid input to run on.

## Verification

**What this cell does:** Three checks. Verification 1 re-confirms the $m=1$/$m=n$ special cases (already checked once above, repeated here as part of the formal verification record) and the $n\geq2f+3$ bound for this lab's $N, F$. Verification 2 checks that under the m30 attack, all three robust methods end Part A's 10-round run strictly closer to $\theta^{*}$ than FedAvg does, and that the Part A m-sweep's robustness panel is smallest at $m=n-f$ and larger once $m$ exceeds it. Verification 3 checks the SecAgg cancellation identity and the LDP $\sigma \propto 1/\epsilon$ relationship, plus that the LDP accuracy-cost curve from Part C decreases (gets closer to the no-DP baseline) as $\epsilon$ grows. All three must print PASS.

In [ ]:
# Verification 1: special cases and the Byzantine-tolerance bound.
assert np.allclose(multi_krum(_V_test, F, m=1)[0], krum(_V_test, F)[0])
assert np.allclose(multi_krum(_V_test, F, m=N)[0], fedavg(_V_test))
assert N >= 2 * F + 3, "This lab's (N, F) must satisfy Krum's n >= 2f+3 guarantee."
print("PASS: multi_krum special cases and the n >= 2f+3 bound hold for this lab's settings.")

# Verification 2: robust methods beat FedAvg under attack; m-sweep breaks past m = n - f.
fedavg_final = results["FedAvg (no defense)"][-1]
for label in ["Krum (m=1)", f"Multi-Krum (m={N-F})", f"Trimmed Mean (beta={F})"]:
    assert results[label][-1] < fedavg_final, f"{label} should end closer to theta* than FedAvg."
best_m_idx = int(np.argmin(attack_curve))
assert m_values[best_m_idx] <= N - F, "Robustness should be best at or before m = n - f."
assert attack_curve[-1] > attack_curve[best_m_idx], "Attack-curve distance should grow once m exceeds n - f."
print("PASS: all three robust methods beat FedAvg under attack, and the m-sweep breaks past m = n - f.")

# Verification 3: SecAgg cancellation identity; LDP sigma and accuracy-cost trends.
assert np.allclose(server_sum, sum_x + sum_b), "Pairwise masks must cancel exactly."
assert not np.allclose(server_sum, sum_x), "Self-masks must NOT cancel on their own."
sigmas = [CLIP_NORM * np.sqrt(2 * np.log(1.25 / DELTA)) / e for e in eps_values]
assert all(sigmas[i] > sigmas[i + 1] for i in range(len(sigmas) - 1)), "sigma must decrease as epsilon grows."
assert ldp_finals[0] >= ldp_finals[-1], "LDP accuracy cost should shrink as epsilon grows (less noise)."
print("PASS: SecAgg cancellation identity and LDP sigma/accuracy trends hold.")

## Optional Challenges

1. Push past the guarantee: set `F = 4` (so `N >= 2*F+3` fails for `N=10`) and have the 4 malicious clients collude by each submitting a gradient close to the *other* malicious clients' gradients rather than a plain scaled honest gradient. Does Krum's score function still pick an honest vector?
2. Implement coordinate-wise **median** (`np.median(V, axis=0)`) as a fifth aggregation rule, add it to the Part A comparison plot, and compare its robustness and convergence speed to Trimmed Mean.
3. Extend the Part B demonstration to simulate one client dropping out after sharing its secrets but before submitting `y_u`. Show what leftover pairwise terms the other clients' submissions now contain, and sketch (in code or in words) what would need to be reconstructed to remove them.
4. Compose Part A and Part C: apply `clip_and_noise` to every honest gradient *and* run the result through Multi-Krum under the m30 attack. Does adding LDP noise change which vectors Multi-Krum selects, compared to the un-noised Part A run with the same attack?

## Assignment:

Extend the experiments in this notebook by modifying the settings below. Complete all four tasks, fill in the results table, and answer the questions below.

**Baseline (what the notebook runs above):** `N=10`, `F=2`, `SCALE=50`, `NUM_ROUNDS=10` for Part A's main comparison; `m_values=[1,2,4,6,8,10]` for the m-sweep; `eps_values=[0.5,1,2,5,10]`, `CLIP_NORM=1.5`, `DELTA=1e-5` for Part C.

Tip: restore each baseline value before moving to the next task. Every task reruns in well under a minute since nothing here retrains a real neural network.

#### Task 1 — Stronger attack ($\Lambda$: 50 → 200)
Restore `F = 2`. Set `SCALE = 200` and re-run the Part A four-way comparison.

*Question:* Does FedAvg's final distance grow roughly in proportion to $\Lambda$? Do Krum, Multi-Krum, and Trimmed Mean's final distances change much at all? Explain using the fact that none of the three robust methods' formulas reference $\Lambda$ directly.

#### Task 2 — More malicious clients, still within the guarantee ($f$: 2 → 3)
Restore `SCALE = 50`. Set `F = 3` (still satisfies $n=10 \geq 2(3)+3=9$) and re-run the Part A comparison, remembering that Multi-Krum's default `m` is `N - F`.

*Question:* How does Krum's final distance at $f=3$ compare to $f=2$? Relate your answer to how many of the $n-f-2$ nearest-neighbor slots in the score formula are now available to honest vectors.

#### Task 3 — Smaller clipping bound ($C$: 1.5 → 0.5)
Restore `F = 2`. Set `CLIP_NORM = 0.5` in Part C and re-run the $\epsilon$ sweep.

*Question:* How does the whole accuracy-cost-vs-$\epsilon$ curve shift compared to the baseline? Using the formula $\sigma = C\sqrt{2\ln(1.25/\delta)}/\epsilon$, explain why a smaller clip bound changes the noise added at every $\epsilon$, not just the clipping itself.

#### Task 4 — Tighter privacy budget range ($\delta$: 1e-5 → 1e-3)
Restore `CLIP_NORM = 1.5`. Set `DELTA = 1e-3` in Part C and re-run the $\epsilon$ sweep.

*Question:* Does loosening $\delta$ (making it a weaker guarantee) increase or decrease $\sigma$ at a fixed $\epsilon$? Does this change match your intuition about $\delta$ as "the probability the $(\epsilon,\delta)$-DP guarantee is allowed to fail outright"?

#### Results table (fill in the final distances)

| Configuration | FedAvg | Krum | Multi-Krum | Trimmed Mean |
|---|---|---|---|---|
| Baseline ($\Lambda$=50, $f$=2) | | | | |
| Task 1 ($\Lambda$=200) | | | | |
| Task 2 ($f$=3) | | | | — |

| Configuration | $\epsilon$=0.5 final dist | $\epsilon$=10 final dist |
|---|---|---|
| Baseline ($C$=1.5, $\delta$=1e-5) | | |
| Task 3 ($C$=0.5) | | |
| Task 4 ($\delta$=1e-3) | | |

#### Written questions

1. FedAvg's final distance scales with $\Lambda$; the three robust methods' do not. State in one sentence why none of `krum_scores`, `multi_krum`, or `trimmed_mean` contains $\Lambda$ anywhere in its formula, and what that implies about how big an attacker's scale factor could be before these defenses would need to change.
2. A deployment is choosing between Krum ($m=1$), Multi-Krum with $m=n-f$, and Trimmed Mean with $\beta=f$. Given your Task 2 results, rank the three by how well they tolerate $f$ approaching its maximum allowed value, and explain the ranking using each rule's formula.
3. Revisit the combined-defense reasoning from the lecture and notes.md: name one pair of the three defenses in this lab (robust aggregation, SecAgg, LDP) that composes cleanly, and one pair that is in tension, and justify each using what you implemented in Parts A-C.

**Deliverables:** this notebook with all cells executed top to bottom, the completed results tables, and your answers to the four task questions plus the three written questions above.